# Exploratory Data Analysis (EDA)

In this notebook, we explore the cleaned academic stress dataset to understand its structure, identify patterns, examine feature distributions, and analyze relationships among variables before developing machine learning models.

# 03 — Exploratory Data Analysis & Target Variable Construction

**Input:**  `data/processed/student_stress_clean.csv`  (32 columns, cleaned data)

**Output:** `data/processed/student_stress_with_target.csv`  (38 columns, with target)

**What this notebook does:**
1. Loads the cleaned survey dataset (39 responses)
2. Checks data shape, types, missing values, duplicates
3. Explores demographic distributions (age, gender, university, program)
4. Reverse-scores the four positively-worded stress questions
5. Computes the total academic stress score (range 10–50)
6. Categorizes students into Low / Moderate / High stress levels (tertile split)
7. Visualizes the target distribution
8. Saves the enriched dataset for the ML stage

**Note:** Only 39 real survey responses exist at this stage. The model built on
this data is a prototype; a larger sample (300+) is required for publication-quality results.

In [ ]:
#Import Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load Dataset

df = pd.read_csv("../data/processed/student_stress_clean.csv")

In [ ]:

# First Look
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(5)

In [ ]:
# Dataset Shape
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")


In [ ]:
# Data Types
df.info()

In [ ]:
# Missing Values
missing = df.isnull().sum()

missing

In [ ]:
missing[missing>0]

In [ ]:
# Duplicate Records
df.duplicated().sum()

In [ ]:
# Summary Statistics
df.describe()

In [ ]:
df.describe(include="object")

In [ ]:
# List Numeric Columns
numeric_columns = df.select_dtypes(include="number").columns

numeric_columns

In [ ]:
# List Categorical Columns
categorical_columns = df.select_dtypes(include="object").columns

categorical_columns

## Observations

- Dataset contains 39 student responses.
- Dataset has 34 variables.
- No missing values were detected.
- No duplicate records were found.
- The dataset contains both categorical and numerical features.
- The dataset is ready for exploratory visualization.

# Exploratory Data Analysis (Part 2)
This is where we generate the figures that will appear in my thesis.

# Age Distribution

In [ ]:


plt.figure(figsize=(8,5))

df["age"].value_counts().plot(kind="bar")

plt.title("Age Distribution of Participants")
plt.xlabel("Age Group")
plt.ylabel("Number of Students")

plt.tight_layout()
plt.show()

### Observation

Most participants belong to the **20–22** age group, followed by **23–25** years. This indicates that the majority of respondents are undergraduate students within the typical university age range.

# Gender Distribution

In [ ]:
plt.figure(figsize=(6,6))

df["gender"].value_counts().plot(
    kind="pie",
    autopct="%1.1f%%"
)

plt.title("Gender Distribution")

plt.ylabel("")

plt.show()

### Observation

The dataset contains responses from both male and female students, providing representation from multiple genders.

# University Distribution

In [ ]:
plt.figure(figsize=(10,5))

df["university"].value_counts().plot(kind="bar")

plt.title("University Distribution")

plt.xlabel("University")

plt.ylabel("Students")

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

Most responses were collected from Tribhuvan University, followed by other universities. This reflects the availability of participants rather than the national distribution of undergraduate students.

# Program Distribution

In [ ]:
plt.figure(figsize=(10,5))

df["program"].value_counts().plot(kind="bar")

plt.title("Program Distribution")

plt.xlabel("Academic Program")

plt.ylabel("Students")

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

### Observation

- Most responses were collected from **BCA students**, followed by students from other undergraduate programs.
- A few responses were received from programs such as BSc CSIT, BE/B.Tech, BBM, Nursing, and others.
- The dataset is not evenly distributed across academic programs because data collection is still in progress.
- A more balanced dataset will improve the generalizability of the machine learning model.

## Stress Score Analysis


In [ ]:
# Step 1: Verify the 10 stress-question columns exist

stress_cols = [f"stress_q{i}" for i in range(1, 11)]
print("Stress columns:")
print(stress_cols)

missing = [c for c in stress_cols if c not in df.columns]
print("\nMissing:", missing)

# Show the first few rows of just these columns
df[stress_cols].head()

### Step 2: Reverse-score the four positively-worded stress questions.
#
### Some stress questions are worded so that a HIGHER score means LESS stress.
### To combine all 10 into a single stress score, we must flip those so that
### every question points in the same direction: higher = more stress.
#
### The formula is: reverse_score = 6 - original_score
#
### Reverse-scored questions: stress_q4, stress_q5, stress_q7, stress_q8



In [ ]:
### List of questions that need reversing
reverse_cols = ["stress_q4", "stress_q5", "stress_q7", "stress_q8"]

### Create NEW columns with "_rev" suffix so we keep the original values
### (this is good practice — we never destroy raw data)
for col in reverse_cols:
    df[col + "_rev"] = 6 - df[col]

### Show a quick comparison: original vs reversed
comparison = df[reverse_cols + [c + "_rev" for c in reverse_cols]]
print("Original vs Reverse-scored values:")
print(comparison.head(10))

### Verify the reversal is correct: max original + max reversed should both be 5
print("\nSanity check:")
for col in reverse_cols:
    orig = df[col]
    rev = df[col + "_rev"]
    print(f"{col}: original range = {orig.min()}-{orig.max()}, reversed range = {rev.min()}-{rev.max()}")

## Step 3 — Total Academic Stress Score

After reverse-scoring the four positively-worded questions, all 10 stress questions
now point in the same direction: **higher = more stress**.

We can therefore combine them into a single number per student — the **total stress score**.

**Columns used in the sum:**
- Original values: `stress_q1`, `stress_q2`, `stress_q3`, `stress_q6`, `stress_q9`, `stress_q10`
- Reversed values: `stress_q4_rev`, `stress_q5_rev`, `stress_q7_rev`, `stress_q8_rev`

**Theoretical range:** 10 (lowest possible) to 50 (highest possible).

A higher score means the student reported more frequent academic stress symptoms
over the past month.

In [ ]:
# Step 3: Build the total academic stress score.
#
# After reverse-scoring, all 10 questions point in the same direction:
# higher = more stress. We can now safely add them up.
#
# Sum uses original values for stress_q1, q2, q3, q6, q9, q10
# and reversed values for stress_q4, q5, q7, q8.

final_stress_cols = [
    "stress_q1",
    "stress_q2",
    "stress_q3",
    "stress_q4_rev",
    "stress_q5_rev",
    "stress_q6",
    "stress_q7_rev",
    "stress_q8_rev",
    "stress_q9",
    "stress_q10",
]

# Create the total stress score column
df["stress_score"] = df[final_stress_cols].sum(axis=1)

# Show the range and a few example rows
print("Stress score range:")
print(f"  Min : {df['stress_score'].min()}")
print(f"  Max : {df['stress_score'].max()}")
print(f"  Mean: {df['stress_score'].mean():.2f}")
print(f"  Std : {df['stress_score'].std():.2f}")

print("\nFirst 10 students' stress scores:")
print(df[["stress_score"]].head(10))

## Step 4 — Distribution of the Total Stress Score

Before categorizing students into stress levels, we examine the distribution
of the total stress score (range 10–50).

The distribution informs:
- Where the center of the sample lies
- Whether scores cluster or spread out
- Where natural cut points might exist for Low / Moderate / High categories

Two plots are used:
1. **Histogram** — shows the frequency of scores across the range
2. **Box plot** — shows median, quartiles, and any outliers

In [ ]:
# Step 4: Visualize the distribution of the stress score.

import matplotlib.pyplot as plt

# ---- Histogram ----
plt.figure(figsize=(9, 5))

plt.hist(
    df["stress_score"],
    bins=10,
    edgecolor="black",
    color="steelblue"
)

plt.title("Distribution of Total Stress Scores (n=39)")
plt.xlabel("Stress Score (10 = lowest, 50 = highest)")
plt.ylabel("Number of Students")

# Draw a vertical line at the mean
mean_score = df["stress_score"].mean()
plt.axvline(mean_score, color="red", linestyle="--", label=f"Mean = {mean_score:.1f}")
plt.legend()

plt.tight_layout()
plt.show()


# ---- Box plot ----
plt.figure(figsize=(7, 4))

plt.boxplot(df["stress_score"], vert=False)

plt.title("Box Plot of Stress Scores")
plt.xlabel("Stress Score")

plt.tight_layout()
plt.show()


# ---- Summary numbers that matter for categorization ----
print("Quartiles of stress_score:")
print(df["stress_score"].describe())

## Step 5 — Defining Stress Categories (ML Target Variable)

The stress score is a continuous number (10–50). For classification-based ML,
we need to convert it into categories.

We use a **tertile split** — dividing the sample into three roughly equal groups
at the 33rd and 67th percentiles. This produces a balanced 3-class target:

- **Low stress** — score at or below the 33rd percentile
- **Moderate stress** — score between 33rd and 67th percentiles
- **High stress** — score above the 67th percentile

This method is:
- **Data-driven** — cut points come from the actual distribution
- **Transparent** — easy to describe in the methodology section
- **Balanced** — each class has approximately equal samples (important for ML)

Students are then assigned a `stress_level` label that becomes the target
variable for the machine learning model.

In [ ]:
# Step 5: Categorize students into Low / Moderate / High stress groups.
#
# We use a tertile split (33rd and 67th percentiles) to create three
# roughly equal groups. This is a standard, defensible approach for
# survey-based classification tasks.

# Compute the cut points
low_cutoff  = df["stress_score"].quantile(0.33)
high_cutoff = df["stress_score"].quantile(0.67)

print(f"Low cutoff  (33rd percentile): {low_cutoff:.2f}")
print(f"High cutoff (67th percentile): {high_cutoff:.2f}")
print()

# Assign categories
def categorize(score):
    if score <= low_cutoff:
        return "Low"
    elif score <= high_cutoff:
        return "Moderate"
    else:
        return "High"

df["stress_level"] = df["stress_score"].apply(categorize)

# Check the class distribution
print("Class distribution:")
print(df["stress_level"].value_counts())
print()
print("Percentages:")
print(df["stress_level"].value_counts(normalize=True).round(2) * 100)

## Step 6 — Save the Updated Dataset

The dataframe now contains two new columns:

- `stress_score` — total stress score (continuous, 10–50)
- `stress_level` — categorical target (Low / Moderate / High)

We save this enriched dataset so that the next stage (feature engineering and ML)
can load a single file containing both features and the target variable.

In [ ]:
# Step 6: Save the updated dataset with stress_score and stress_level
#
# This becomes the input to the next stage: feature engineering.

import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/student_stress_with_target.csv", index=False)

print("Saved: ../data/processed/student_stress_with_target.csv")
print("Shape :", df.shape)
print("Columns:", df.columns.tolist())

## Step 7 — Distribution of Stress Categories

The figure below shows how students are distributed across the three
stress categories (Low / Moderate / High). This confirms the class balance
before ML training.

A balanced target distribution reduces the risk of the model being biased
toward one class.

In [ ]:
# Step 7: Visualize the stress_level class distribution

import matplotlib.pyplot as plt

# Count in a consistent order
order = ["Low", "Moderate", "High"]
counts = df["stress_level"].value_counts().reindex(order)

plt.figure(figsize=(7, 5))

bars = plt.bar(counts.index, counts.values, color=["#4CAF50", "#FFC107", "#F44336"], edgecolor="black")

# Add count labels on top of each bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.2, str(int(height)),
             ha="center", fontsize=11)

plt.title("Distribution of Academic Stress Levels (n=39)")
plt.xlabel("Stress Level")
plt.ylabel("Number of Students")
plt.ylim(0, max(counts.values) + 2)

plt.tight_layout()
plt.show()